<a href="https://colab.research.google.com/github/jinjtp/SuperAI-SS-6-RAG-MiniHackathon/blob/main/KBTG_605345_FahMai_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Starter Kit FahMai RAG

This notebook walks through building a minimalist **Retrieval-Augmented Generation (RAG)** system for the FahMai electronics store Q&A challenge.

**Sections:**
- **Section 0:** Setup & LLM Test
- **Section 1:** Dense Retrieval (MiniLM)
- **Section 2:** Sparse Retrieval (BM25)
- **Section 3:** Hybrid Retrieval (RRF)
- **Section 4:** What's Next?

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# === CONFIGURATION ===
# Change N_QUESTIONS to 100 for a full competition run.

N_QUESTIONS = 100  # 10 for demo, 100 for real submission
DATA_DIR = "/content/drive/MyDrive/fahmai_data"
KB_DIR = f"/content/drive/MyDrive/fahmai_data/knowledge_base"

---
## Section 0: Setup & LLM Test

First, install dependencies and test the ThaiLLM API — **without** any retrieval. This shows why RAG is needed.

ThaiLLM website: https://playground.thaillm.or.th/chat/

In [ ]:
!pip install -q sentence-transformers pythainlp rank-bm25 requests python-dotenv langchain langchain-text-splitters

In [ ]:
import os, csv, re, time, requests
import numpy as np
from pathlib import Path
from google.colab import userdata

THAILLM_API_KEY = userdata.get('ThaiLLM')

In [ ]:
def ask_llm(messages, model="kbtg", max_retries=5):
    """Call ThaiLLM API with retry and rate-limit handling.

    Available models: typhoon, openthaigpt, pathumma, kbtg
    """
    url = f"http://thaillm.or.th/api/{model}/v1/chat/completions"
    headers = {"Content-Type": "application/json", "apikey": THAILLM_API_KEY}
    payload = {
        "model": "/model",
        "messages": messages,
        "max_tokens": 2024,
        "temperature": 0,
    }

    for attempt in range(max_retries):
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=120)

            # Rate limit — wait and retry
            if resp.status_code == 429:
                wait = min(2 ** attempt, 30)
                print(f"  Rate limited, waiting {wait}s...")
                time.sleep(wait)
                continue

            resp.raise_for_status()
            return resp.json()["choices"][0]["message"]["content"].strip()

        except requests.exceptions.RequestException as e:
            wait = 2 ** attempt
            print(f"  Error: {e}, retrying in {wait}s...")
            time.sleep(wait)

    return None


def parse_answer(text):
    """Extract answer number from LLM response."""
    if text is None:
        return None
    # Remove any <think>...</think> blocks (some models do chain-of-thought)
    clean = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    # Look for ANSWER: X pattern
    m = re.search(r"ANSWER:\s*(\d+)", clean)
    if m:
        return int(m.group(1))
    # Fallback: first standalone number 1-10
    for d in re.findall(r"\b(\d{1,2})\b", clean):
        if 1 <= int(d) <= 10:
            return int(d)
    return None

### Test the LLM without RAG

Let's ask a FahMai-specific question *without* any context. The LLM shouldn't know the answer.

In [ ]:
# Ask without context — LLM has no idea about FahMai's products
response = ask_llm([
    {"role": "user", "content": "Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?"}
])
print("LLM response (no context):", response)
print("\n→ The LLM doesn't know FahMai-specific facts. We need RAG!")

LLM response (no context): <think>
Okay, the user is asking about the water resistance rating of the Samsung Galaxy S3 Ultra, specifically how many ATM it can withstand. First, I need to recall the specifications of the S3 Ultra. I remember that the S3 Ultra is a variant of the Galaxy S3, but I'm not entirely sure about its water resistance. 

Wait, the original Galaxy S3 was not water-resistant. I think the S3 Ultra might have some water resistance, but I need to confirm. Let me check. The S3 Ultra was released in 2013, and at that time, water resistance wasn't a common feature in smartphones. However, some models might have had IP ratings. 

Looking up the specifications, the S3 Ultra has an IP67 rating. IP67 means it's dust-tight and can withstand immersion in water up to 1 meter for 30 minutes. To convert that to ATM, I need to know that 1 meter of water is approximately 1 ATM. So, 1 meter is 1 ATM, and 30 minutes is the duration. Therefore, the S3 Ultra is rated for 1 ATM. 

But w

### Load Questions

In [ ]:
questions = []
with open(f"{DATA_DIR}/questions.csv", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        choices = {str(i): row[f"choice_{i}"] for i in range(1, 11)}
        questions.append({"id": int(row["id"]), "question": row["question"], "choices": choices})

print(f"Loaded {len(questions)} questions, using first {N_QUESTIONS} for this run")
print(f"\nExample — Q1: {questions[0]['question']}")
for k, v in questions[0]["choices"].items():
    print(f"  {k}. {v}")

Loaded 100 questions, using first 100 for this run

Example — Q1: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ
  1. 3 ATM
  2. IP68
  3. 5 ATM
  4. IP67
  5. 10 ATM
  6. 20 ATM
  7. IPX8
  8. 1 ATM
  9. ไม่มีข้อมูลนี้ในฐานข้อมูล
  10. คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า


---
## Section 1: Dense Retrieval (MiniLM)

**Dense retrieval** converts text into vectors (embeddings) and finds relevant chunks by cosine similarity.

The pipeline: **Load docs → Chunk → Embed → Retrieve → Generate**

### 1.1 Load Knowledge Base

In [ ]:
kb_dir = Path(KB_DIR)
documents = []
for fp in sorted(kb_dir.rglob("*.md")):
    text = fp.read_text(encoding="utf-8")
    documents.append({"path": str(fp.relative_to(kb_dir)), "text": text})

print(f"Loaded {len(documents)} documents")
print(f"  products/: {sum(1 for d in documents if 'products/' in d['path'])}")
print(f"  policies/: {sum(1 for d in documents if 'policies/' in d['path'])}")
print(f"  store_info/: {sum(1 for d in documents if 'store_info/' in d['path'])}")

# Preview one document
print(f"\n--- Sample: {documents[0]['path']} ---")
print(documents[0]["text"][:500])

Loaded 118 documents
  products/: 110
  policies/: 5
  store_info/: 3

--- Sample: policies/cancellation_policy.md ---
# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่

**วันที่อัปเดต:** 1 มีนาคม 2569

---

## 1. ภาพรวมนโยบาย

ฟ้าใหม่เข้าใจว่าบางครั้งลูกค้าอาจต้องการยกเลิกคำสั่งซื้อด้วยเหตุผลต่างๆ นโยบายนี้อธิบายสิทธิ์และขั้นตอนการยกเลิกคำสั่งซื้อตามสถานะของคำสั่งซื้อในขณะนั้น ความสามารถในการยกเลิกขึ้นอยู่กับสถานะคำสั่งซื้อเป็นหลัก

---

## 2. การยกเลิกตามสถานะคำสั่งซื้อ

### 2.1 สถานะ "รอชำระเงิน" (Pending Payment)

**ยกเลิกได้ทันที**

คำสั่งซื้อที่อยู่ในสถานะรอชำระเงินสามารถยกเลิกได้ทันทีโดยไม่มีค่าใช้จ่าย ผ่านแอปพลิ


### 1.2 Chunking

LLMs have limited context windows, and long documents dilute relevance. We split each document into smaller **chunks** using a fixed-size sliding window with overlap.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

CHUNK_SIZE = 256
CHUNK_OVERLAP = 50

# 1. Set up the Smart Splitter
# This tries to split by double-enter (\n\n) first.
# If a paragraph is still bigger than 512, it tries single-enter (\n), then spaces.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", " ", ""]
)

chunks = []

# 2. Build all chunks structurally
for doc in documents:
    # Get your metadata (like you already did!)
    filename = os.path.basename(doc["path"]).replace(".md", "").replace("_", " ")

    # Split the document text structurally
    doc_chunks = text_splitter.split_text(doc["text"])

    for window in doc_chunks:
        # Re-inject your metadata into the newly structured chunk
        enriched_text = f"ชื่อสินค้า/หมวดหมู่: {filename}\nรายละเอียด: {window}"

        chunks.append({
            "text": enriched_text,
            "source": doc["path"]
        })

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\n--- Sample chunk ---")
print(f"Source: {chunks[0]['source']}")
print(chunks[0]["text"][:300])

Created 2225 chunks from 118 documents

--- Sample chunk ---
Source: policies/cancellation_policy.md
ชื่อสินค้า/หมวดหมู่: cancellation policy
รายละเอียด: # นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่

**วันที่อัปเดต:** 1 มีนาคม 2569

---

## 1. ภาพรวมนโยบาย


### 1.3 Embedding

We use `BAAI/bge-m3` to generate vector embeddings while added cross encoder to ensure even higher level of accuracy.

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("BAAI/bge-m3")

chunk_texts = [c["text"] for c in chunks]
chunk_embeddings = embed_model.encode(
    chunk_texts,
    batch_size= 32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(f"Chunk embeddings shape: {chunk_embeddings.shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/140 [00:00<?, ?it/s]

Chunk embeddings shape: (2225, 1024)


In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

### 1.4 Retrieve

Embed the question, then find the most similar chunks via dot product (= cosine similarity for normalized vectors).

In [ ]:
TOP_K = 5

def dense_retrieve(query, chunk_embs, k=TOP_K):
    """Return indices of top-k most similar chunks to the query."""
    q_emb = embed_model.encode([query], normalize_embeddings=True)
    scores = np.dot(chunk_embs, q_emb.T).flatten()  # cosine similarity (vectors are normalized)
    top_idx = np.argsort(scores)[::-1][:k]
    return top_idx, scores[top_idx]

# Demo: retrieve for Q1
q = questions[0]
idx, scores = dense_retrieve(q["question"], chunk_embeddings)

print(f"Question: {q['question']}\n")
for rank, (i, s) in enumerate(zip(idx, scores), 1):
    print(f"  Rank {rank} (score={s:.3f}) [{chunks[i]['source']}]")
    print(f"  {chunks[i]['text'][:150]}...")
    print()

Question: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ

  Rank 1 (score=0.725) [products/WK-SW-001_wongkhojon_watch_s3_ultra.md]
  ชื่อสินค้า/หมวดหมู่: WK-SW-001 wongkhojon watch s3 ultra
รายละเอียด: ความโดดเด่นที่สุดของ S3 Ultra คือมาตรฐานกันน้ำระดับ 100 เมตร (10 ATM) พร้อมโหมด D...

  Rank 2 (score=0.685) [products/WK-SW-001_wongkhojon_watch_s3_ultra.md]
  ชื่อสินค้า/หมวดหมู่: WK-SW-001 wongkhojon watch s3 ultra
รายละเอียด: A: Dive Mode ของ S3 Ultra รองรับการดำน้ำสูงสุด 40 เมตร (ลึกกว่านั้นนาฬิกาจะไม่รับ...

  Rank 3 (score=0.673) [products/WK-SW-002_wongkhojon_watch_s3_pro.md]
  ชื่อสินค้า/หมวดหมู่: WK-SW-002 wongkhojon watch s3 pro
รายละเอียด: **Q: ว่ายน้ำได้ไหมคะ? ดำน้ำได้ด้วยไหม?**
A: ว่ายน้ำได้สบายเลยค่ะ กันน้ำ 5 ATM (50 เ...

  Rank 4 (score=0.673) [products/WK-SW-001_wongkhojon_watch_s3_ultra.md]
  ชื่อสินค้า/หมวดหมู่: WK-SW-001 wongkhojon watch s3 ultra
รายละเอียด: A: S3 Ultra แตกต่างจาก S3 Pro ในหลายด้านครับ ได้แก่ (1) ตัวเรือนไทเทเนียม เบากว่า...

  Rank 5 (score=0.635) [products/W

### 1.5 Generate Answer

Send the retrieved context + question + choices to the LLM and parse the answer.

In [ ]:
# A very basic system prompt — you should improve this!
#keeping the og prompt in case mine fails
#SYSTEM_PROMPT = "ตอบคำถามจากข้อมูลที่ให้มา เลือกตัวเลือกที่ถูกต้องที่สุด ตอบเป็น ANSWER: X เท่านั้น"
SYSTEM_PROMPT = """คุณคือผู้ช่วย AI คนใหม่ของร้านฟ้าใหม่ ร้านอุปกรณ์อิเล็คทรอนิกส์ของไทย
หน้าที่ของคุณคือการตอบคำถามที่ลูกค้าถามมาโดยอ้างอิงกับข้อมูลที่ร้านให้มาเท่านั้น
ห้ามหาข้อมูลจากแหล่งอื่นหรือเดาคำตอบขึ้นมาเองเด็ดขาด กฎการตอบคำถาม:
1. วิเคราะห์ข้อมูลอ้างอิงและตัวเลือกอย่างละเอียดเพื่อหาตัวเลือกที่ถูกต้องที่สุดเพียงข้อเดียว
2. หากในข้อมูลอ้างอิงไม่มีข้อมูลที่สามารถตอบคำถามนี้ได้เลย ให้เลือกตัวเลือกที่ 9
3. หากคำถามเป็นเรื่องทั่วไปที่ไม่เกี่ยวข้องกับสินค้า บริการ นโยบาย หรือร้านฟ้าใหม่เลย ให้เลือกตัวเลือกที่ 10
4. ตอบเป็นตัวเลขเท่านั้น ห้ามตอบอย่างอื่น"""

def build_rag_prompt(question, choices, retrieved_chunks):
    """Build the user prompt with retrieved context.

    TODO: Design your own prompt format.
    Think about: How should you present the context? The choices?
    What instructions help the LLM pick the right answer?
    """
    context = "\n\n".join(
        f"--- Chunk {i+1} ---\n{c['text']}"
        for i, c in enumerate(retrieved_chunks)
    )
    choices_text = "\n".join(f"{k}. {v}" for k, v in choices.items())
    # === CUSTOMIZE THIS PROMPT ===
    return (
        f"{context}\n\n"
        f"คำถาม: {question}\n\n"
        f"ตัวเลือก:\n{choices_text}\n\n"
        f"ตอบ ANSWER: X (X คือหมายเลขตัวเลือก 1-10)"
    )

# Demo: answer Q1
q = questions[0]
idx, _ = dense_retrieve(q["question"], chunk_embeddings)
retrieved = [chunks[i] for i in idx]

prompt = build_rag_prompt(q["question"], q["choices"], retrieved)
raw = ask_llm([
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": prompt},
])
answer = parse_answer(raw)
print(f"Q{q['id']}: {q['question']}")
print(f"LLM raw: {raw}")
print(f"Parsed answer: {answer}")

Q1: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ
LLM raw: <think>
Okay, let's tackle this question. The user is asking about the water resistance of the Watch S3 Ultra in ATM. I need to find the correct answer by checking the provided chunks.

Looking at Chunk 1, it mentions that the S3 Ultra has a water resistance standard of 100 meters (10 ATM) and includes a Dive Mode. Chunk 2 also states that the Dive Mode supports diving up to 40 meters, and it's important to note that going deeper than that might void the warranty. Chunk 4 compares S3 Ultra with S3 Pro, highlighting that S3 Ultra is water-resistant at 100 meters (10 ATM) with Dive Mode, while S3 Pro is only 50 meters (5 ATM). 

So, from these chunks, it's clear that the S3 Ultra has a water resistance of 10 ATM. The other options like 5 ATM or 3 ATM are mentioned for the S3 Pro, not the Ultra. Therefore, the correct answer should be option 5: 10 ATM.
</think>

5
Parsed answer: 5


In [ ]:
retrieved

[{'text': 'ชื่อสินค้า/หมวดหมู่: WK-SW-001 wongkhojon watch s3 ultra\nรายละเอียด: ความโดดเด่นที่สุดของ S3 Ultra คือมาตรฐานกันน้ำระดับ 100 เมตร (10 ATM) พร้อมโหมด Dive Mode ที่รองรับการดำน้ำได้ถึง 40 เมตร บันทึกความลึก ความดัน อุณหภูมิน้ำ และเวลาดำน้ำแบบ real-time แสดงผลบนหน้าจอ AMOLED 1.9 นิ้วที่ยังคงอ่านได้ชัดเจนใต้น้ำ',
  'source': 'products/WK-SW-001_wongkhojon_watch_s3_ultra.md'},
 {'text': 'ชื่อสินค้า/หมวดหมู่: WK-SW-001 wongkhojon watch s3 ultra\nรายละเอียด: A: Dive Mode ของ S3 Ultra รองรับการดำน้ำสูงสุด 40 เมตร (ลึกกว่านั้นนาฬิกาจะไม่รับประกันความถูกต้องของข้อมูลและอาจเกิดความเสียหายได้) ในระหว่างดำน้ำจะแสดงผลความลึก อุณหภูมิน้ำ และเวลาดำน้ำ แต่ขอแนะนำว่าไม่ควรดำน้ำในน้ำร้อน น้ำพุร้อน หรือน้ำทะเลที่มีแรงดัน',
  'source': 'products/WK-SW-001_wongkhojon_watch_s3_ultra.md'},
 {'text': 'ชื่อสินค้า/หมวดหมู่: WK-SW-002 wongkhojon watch s3 pro\nรายละเอียด: **Q: ว่ายน้ำได้ไหมคะ? ดำน้ำได้ด้วยไหม?**\nA: ว่ายน้ำได้สบายเลยค่ะ กันน้ำ 5 ATM (50 เมตร) รองรับว่ายน้ำในสระและทะเล แต่ไม่มีโหมด Dive

### 1.6 Run All Questions (Dense)

Loop through questions, retrieve, generate, and collect predictions.

In [ ]:
def run_pipeline(questions, retrieve_fn, label="dense", n=N_QUESTIONS):
    """Run retrieval + LLM on first n questions. Returns predictions dict."""
    predictions = {}
    for i, q in enumerate(questions[:n]):
        retrieved_chunks = retrieve_fn(q["question"])
        prompt = build_rag_prompt(q["question"], q["choices"], retrieved_chunks)
        raw = ask_llm([
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ])
        pred = parse_answer(raw)
        predictions[q["id"]] = pred if pred else 1  # default to 1 if parse fails
        print(f"  Q{q['id']:>3}: pred={predictions[q['id']]}")
        time.sleep(0.3)  # be nice to the API
    print(f"\n{label}: answered {len(predictions)} questions")
    return predictions


  Q  1: pred=5
  Q  2: pred=7
  Q  3: pred=2
  Q  4: pred=6
  Q  5: pred=6
  Q  6: pred=8
  Q  7: pred=1
  Q  8: pred=4
  Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/kbtg/v1/chat/completions, retrying in 1s...
  Q  9: pred=4
  Q 10: pred=2
  Q 11: pred=4
  Q 12: pred=1
  Q 13: pred=2
  Q 14: pred=8
  Q 15: pred=7
  Q 16: pred=1
  Q 17: pred=8
  Q 18: pred=5
  Q 19: pred=2
  Q 20: pred=2
  Q 21: pred=8
  Q 22: pred=3
  Q 23: pred=9
  Q 24: pred=9
  Q 25: pred=5
  Q 26: pred=6
  Q 27: pred=2
  Q 28: pred=7
  Q 29: pred=4
  Q 30: pred=3
  Q 31: pred=9
  Q 32: pred=9
  Q 33: pred=8
  Q 34: pred=5
  Q 35: pred=3
  Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/kbtg/v1/chat/completions, retrying in 1s...
  Q 36: pred=2
  Q 37: pred=8
  Q 38: pred=6
  Q 39: pred=4
  Q 40: pred=9
  Q 41: pred=7
  Q 42: pred=2
  Q 43: pred=4
  Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/kbtg/v1/chat/completions, retrying in 1s...
  Q 

In [ ]:
'''
# Dense retrieval function
def dense_retrieve_chunks(query):
    idx, _ = dense_retrieve(query, chunk_embeddings)
    return [chunks[i] for i in idx]

dense_preds = run_pipeline(questions, dense_retrieve_chunks, label="dense")
'''

---
## Section 2: Sparse Retrieval (BM25)

**BM25** is a keyword-matching algorithm. It scores documents by how many query terms they contain, weighted by term rarity (IDF). No neural network needed.

### 2.1 Thai Tokenization

BM25 needs tokenized text. Thai has no spaces between words, so we use `pythainlp` to segment.

In [ ]:
from pythainlp.tokenize import word_tokenize

# Demo: tokenize a Thai sentence
sample = "Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?"
tokens = word_tokenize(sample, engine="newmm")
print(f"Input:  {sample}")
print(f"Tokens: {tokens}")

Input:  Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?
Tokens: ['Watch', ' ', 'S', '3', ' ', 'Ultra', ' ', 'กันน้ำ', 'ได้', 'กี่', ' ', 'ATM', ' ', 'ครับ', '?']


### 2.2 Build BM25 Index

In [ ]:
from rank_bm25 import BM25Okapi

# Tokenize all chunks
tokenized_chunks = [word_tokenize(c["text"], engine="newmm") for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)

print(f"BM25 index built over {len(tokenized_chunks)} chunks")

BM25 index built over 2225 chunks


### 2.3 Retrieve with BM25

Compare BM25 results with dense results for the same question.

In [ ]:
'''
def bm25_retrieve(query, k=TOP_K):
    """Return top-k chunk indices using BM25."""
    tokens = word_tokenize(query, engine="newmm")
    scores = bm25.get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:k]
    return top_idx, scores[top_idx]

# Compare: same question, two retrieval methods
q = questions[0]
print(f"Question: {q['question']}\n")

d_idx, _ = dense_retrieve(q["question"], chunk_embeddings)
b_idx, _ = bm25_retrieve(q["question"])

print("Dense top-5 sources:")
for i in d_idx:
    print(f"  {chunks[i]['source']}")

print("\nBM25 top-5 sources:")
for i in b_idx:
    print(f"  {chunks[i]['source']}")
'''

Question: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ

Dense top-5 sources:
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-002_wongkhojon_watch_s3_pro.md
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md

BM25 top-5 sources:
  products/WK-SW-002_wongkhojon_watch_s3_pro.md
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-008_wongkhojon_watch_s2_pro.md


### 2.4 Run All Questions (BM25)

In [ ]:
def bm25_retrieve_chunks(query):
    idx, _ = bm25_retrieve(query)
    return [chunks[i] for i in idx]

bm25_preds = run_pipeline(questions, bm25_retrieve_chunks, label="bm25")

---
## Section 3: Hybrid Retrieval (RRF)

**Hybrid** combines dense and sparse results. The idea: dense is good at semantic matching, BM25 is good at exact keyword matching. Together they cover more cases.

We use **Reciprocal Rank Fusion (RRF)** to merge the two ranked lists:

$$\text{score}(d) = \sum_{r \in \text{rankers}} \frac{1}{k + \text{rank}_r(d)}$$

where $k$ is a constant (typically 60). Documents ranked highly by *either* method get a high combined score.

In [ ]:
def hybrid_retrieve(query, chunk_embs, k=TOP_K, rrf_k=60):
    """Combine dense + BM25 results using Reciprocal Rank Fusion."""
    # Get top candidates from each method (fetch more than k to improve fusion)
    fetch_k = k * 2
    d_idx, _ = dense_retrieve(query, chunk_embs, k=fetch_k)
    b_idx, _ = bm25_retrieve(query, k=fetch_k)

    # Compute RRF scores
    rrf_scores = {}
    for rank, idx in enumerate(d_idx, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (rrf_k + rank)
    for rank, idx in enumerate(b_idx, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (rrf_k + rank)

    # Sort by combined score, return top-k
    sorted_idx = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)[:k]
    return sorted_idx

def hybrid_rerank_retrieve_chunks(query, fetch_k=15, final_k=5):
    """Fetches top 15 chunks via Hybrid, then reranks down to the best 5."""
    # This line uses the function above!
    candidate_idx = hybrid_retrieve(query, chunk_embeddings, k=fetch_k)
    candidate_chunks = [chunks[i] for i in candidate_idx]

    pairs = [[query, chunk["text"]] for chunk in candidate_chunks]
    scores = reranker.predict(pairs)

    scored_chunks = list(zip(candidate_chunks, scores))
    scored_chunks.sort(key=lambda x: x[1], reverse=True)

    best_chunks = [chunk[0] for chunk in scored_chunks[:final_k]]
    return best_chunks

'''
# Demo
q = questions[0]
h_idx = hybrid_retrieve(q["question"], chunk_embeddings)
print(f"Question: {q['question']}\n")
print("Hybrid top-5 sources:")
for i in h_idx:
    print(f"  {chunks[i]['source']}")
'''
# Demo: Test the new Hybrid + Reranker on Question 1
q = questions[0]
best_chunks = hybrid_rerank_retrieve_chunks(q["question"])

print(f"Question: {q['question']}\n")
print("Hybrid + Rerank top-5 sources:")
for chunk in best_chunks:
    print(f"  Source: {chunk['source']}")
    print(f"  Snippet: {chunk['text'][:100]}...\n")

Question: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ

Hybrid + Rerank top-5 sources:
  Source: products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  Snippet: ชื่อสินค้า/หมวดหมู่: WK-SW-001 wongkhojon watch s3 ultra
รายละเอียด: ความโดดเด่นที่สุดของ S3 Ultra ค...

  Source: products/WK-SW-002_wongkhojon_watch_s3_pro.md
  Snippet: ชื่อสินค้า/หมวดหมู่: WK-SW-002 wongkhojon watch s3 pro
รายละเอียด: **Q: ว่ายน้ำได้ไหมคะ? ดำน้ำได้ด้ว...

  Source: products/WK-SW-003_wongkhojon_watch_s3.md
  Snippet: ชื่อสินค้า/หมวดหมู่: WK-SW-003 wongkhojon watch s3
รายละเอียด: ---

## รายละเอียดสินค้า

สำหรับผู้ที...

  Source: products/WK-SW-002_wongkhojon_watch_s3_pro.md
  Snippet: ชื่อสินค้า/หมวดหมู่: WK-SW-002 wongkhojon watch s3 pro
รายละเอียด: ปั่นจักรยาน ว่ายน้ำ และเดินป่า มา...

  Source: products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  Snippet: ชื่อสินค้า/หมวดหมู่: WK-SW-001 wongkhojon watch s3 ultra
รายละเอียด: A: S3 Ultra แตกต่างจาก S3 Pro ใ...



### 3.2 Run All Questions (Hybrid)

In [ ]:
def hybrid_retrieve_chunks(query):
    idx = hybrid_retrieve(query, chunk_embeddings)
    return [chunks[i] for i in idx]

'''
hybrid_preds = run_pipeline(questions, hybrid_retrieve_chunks, label="hybrid")
'''
rerank_preds = run_pipeline(questions, hybrid_rerank_retrieve_chunks, label="hybrid+rerank")

  Q  1: pred=5
  Q  2: pred=7
  Q  3: pred=2
  Q  4: pred=6
  Q  5: pred=6
  Q  6: pred=8
  Q  7: pred=1
  Q  8: pred=4
  Q  9: pred=4
  Q 10: pred=7
  Q 11: pred=4
  Q 12: pred=1
  Q 13: pred=6
  Q 14: pred=4
  Q 15: pred=7
  Q 16: pred=1
  Q 17: pred=8
  Q 18: pred=5
  Q 19: pred=2
  Q 20: pred=2
  Q 21: pred=3
  Q 22: pred=9
  Q 23: pred=9
  Q 24: pred=3
  Q 25: pred=5
  Q 26: pred=6
  Q 27: pred=2
  Q 28: pred=7
  Q 29: pred=4
  Q 30: pred=3
  Q 31: pred=4
  Q 32: pred=9
  Q 33: pred=8
  Q 34: pred=5
  Q 35: pred=3
  Q 36: pred=2
  Q 37: pred=8
  Q 38: pred=6
  Q 39: pred=4
  Q 40: pred=9
  Q 41: pred=7
  Q 42: pred=2
  Q 43: pred=4
  Q 44: pred=1
  Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/kbtg/v1/chat/completions, retrying in 1s...
  Q 45: pred=3
  Q 46: pred=1
  Q 47: pred=2
  Q 48: pred=2
  Q 49: pred=6
  Q 50: pred=5
  Q 51: pred=7
  Q 52: pred=4
  Q 53: pred=9
  Q 54: pred=9
  Q 55: pred=9
  Q 56: pred=9
  Q 57: pred=9
  Q 58: pred=9
  Q 59: pred=

### 3.3 Compare All Three Methods

In [ ]:
print(f"{'QID':>4}  {'Dense':>6} {'BM25':>6} {'Hybrid':>7}")
print("-" * 30)
for q in questions[:N_QUESTIONS]:
    qid = q["id"]
    d = dense_preds.get(qid, "-")
    b = bm25_preds.get(qid, "-")
    h = rerank_preds.get(qid, "-")
    match = "" if d == b == h else "  ← differ"
    print(f"Q{qid:>3}  {d:>6} {b:>6} {h:>7}{match}")

### Write Submission

Pick your best method and generate a `submission.csv` for Kaggle.

> Set `N_QUESTIONS = 100` at the top and re-run the notebook to generate a full submission.

In [ ]:
# Use dense predictions as the submission (change to hybrid_preds or bm25_preds to try others)
best_preds =rerank_preds

with open("submission.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "answer"])
    for q in questions:
        qid = q["id"]
        writer.writerow([qid, best_preds.get(qid, 1)])  # default=1 for unanswered

print(f"submission.csv written ({len(questions)} rows)")

submission.csv written (100 rows)


---
## Section 4: What's Next?

This baseline uses a simple setup. Here are ideas to improve your score:

- **Better embeddings** — try other, stronger multilingual  embedding.
- **Smarter chunking** — split by structure or other methods or add useful information to each chunk
- **Chunk size tuning** — experiment with  256, 512, 1024 or something else character chunks
- **Different ThaiLLMs** — try `openthaigpt`, `kbtg`, `pathumma`.
- **Prompt engineering** — adjust the system prompt, add few-shot examples, or change the output format
- **Reranking** — use a cross-encoder or specialized reranker to re-score the top-k chunks before sending to the

**Feel free to implement your own RAG.**